# Simulador: Flexura litostática

## Instalación de componentes visuales

Página de referencia: 
- https://pypi.org/project/ipywidgets/ 

In [1]:
!pip install -U ipywidgets

## Bibliotecas y constantes utilizadas

Páginas de referencia:
- https://numpy.org
- https://matplotlib.org
- https://pypi.org/project/ipywidgets/

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import FormatStrFormatter, MultipleLocator
from ipywidgets import interact, FloatLogSlider, FloatSlider, IntSlider
import ipywidgets as widgets
from IPython.display import display, clear_output

In [3]:
#widgets.Widget.widget_types

In [4]:
# -----------------------------
# Constante
# -----------------------------
g = 9.81  # gravedad [m/s^2]

## Controladores visuales

In [5]:
# -----------------------------
# Controladores
# -----------------------------
slider_D = widgets.FloatLogSlider(
    value=1e23,
    base=10,
    min=21,
    max=25,
    step=0.1,
    description='Rigidez flexural [N·m]',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='580px'),
    continuous_update=False
)

slider_V0 = widgets.FloatLogSlider(
    value=1e13,
    base=10,
    min=11,
    max=14,
    step=0.1,
    description='Peso [N]',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='580px'),
    continuous_update=False
)

slider_rho_m = widgets.FloatSlider(
    value=3300,
    min=3000,
    max=3500,
    step=25,
    description='ρ del manto [kg/m^3]',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='580px'),
    continuous_update=False
)

slider_rho_l = widgets.FloatSlider(
    value=2600,
    min=2000,
    max=3000,
    step=25,
    description='ρ de sedimentos [kg/m^3]',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='580px'),
    continuous_update=False
)

slider_xmax = widgets.IntSlider(
    value=600,
    min=100,
    max=2000,
    step=50,
    description='Inicio del backbulge [km]',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='580px'),
    continuous_update=False
)

slider_npts = widgets.IntSlider(
    value=1200,
    min=200,
    max=5000,
    step=100,
    description='Resolución',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='580px'),
    continuous_update=False
)

check_forebulge = widgets.Checkbox(
    value=True,
    description='Mostrar parámetros geométricos',
    indent=False
)

check_sombreado = widgets.Checkbox(
    value=True,
    description='Sombrear foredeep',
    indent=False
)

check_texto = widgets.Checkbox(
    value=True,
    description='Mostrar cuadro de parámetros',
    indent=False
)

boton_reset = widgets.Button(
    description='Restablecer valores',
    button_style='warning',
    icon='refresh',
    layout=widgets.Layout(width='200px')
)

# -----------------------------
# Marco de salida
# -----------------------------
out = widgets.Output()

## Función que describe la geometría de la flexura litosférica

In [6]:
# -----------------------------
# Función de la flexura litosférica
# -----------------------------
def actualizar(_=None):
    out.clear_output(wait=True)

    with out:
        D = slider_D.value
        V0 = slider_V0.value
        rho_m = slider_rho_m.value
        rho_l = slider_rho_l.value
        x_max_km = slider_xmax.value
        npts = slider_npts.value

        delta_rho = rho_m - rho_l

        # Parámetros flexurales
        alpha = (4 * D / (delta_rho * g)) ** 0.25
        wmax = V0 / (2 * delta_rho * g * alpha)

        # Parámetros geométricos
        x0 = 3 * np.pi * alpha / 4
        xb = np.pi * alpha
        x0p = 7 * np.pi * alpha / 4
        wb = -0.0432 * wmax

        # Dominio espacial
        x_km = np.linspace(0, x_max_km, npts)
        x_m = x_km * 1e3

        # Solución
        xa = np.abs(x_m) / alpha
        w_m = -wmax * np.exp(-xa) * (np.cos(xa) + np.sin(xa))

        # Conversiones a km
        w_km = w_m / 1e3
        alpha_km = alpha / 1e3
        x0_km = x0 / 1e3
        xb_km = xb / 1e3
        x0p_km = x0p / 1e3
        wmax_km = wmax / 1e3
        wb_km = wb / 1e3

        # -----------------------------
        # Figura
        # -----------------------------
        fig,ax = plt.subplots(figsize=(12, 6))

        ax.plot(x_km, w_km, linewidth=2.5, label='Curva de flexura')

        # Formato de y
        ax.yaxis.set_major_locator(ticker.MaxNLocator(8))  
        #ax.yaxis.set_major_locator(MultipleLocator(0.5))
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

        # Línea base
        ax.axhline(0, color='black', linewidth=1.0)
        ax.axvline(0, color='gray', linestyle='--', linewidth=1.9, label='Carga')

        # Sombreado del foredeep
        if check_sombreado.value:
            ax.fill_between(x_km, w_km, 0,  color='brown',where=(w_km < 0), alpha=0.25, label='Foredeep')

        # Líneas geométricas
        if check_forebulge.value:
            if x0_km <= x_max_km:
                ax.axvline(x0_km, linestyle=':', color='blue', linewidth=1.8, label='Inicio de forebulge')
            if xb_km <= x_max_km:
                ax.axvline(xb_km, linestyle='--', color='pink', linewidth=1.8, label='Cresta de forebulge')
            if x0p_km <= x_max_km:
                ax.axvline(x0p_km, linestyle='-.', color='green', linewidth=1.8, label="Fin de forebulge'")

        # Etiquetas y formato
        ax.set_title('Simulador de flexura litosférica', fontsize=15, weight='bold')
        ax.set_xlabel('Distancia desde la carga (km)', fontsize=12)
        ax.set_ylabel('Deflexión w (km)', fontsize=12)
        ax.grid(True, alpha=0.25)
        ax.legend(loc='best', fontsize=10)

        # Cuadro de texto con resultados
        if check_texto.value:
            texto = (
                f"Δρ = {delta_rho:.1f} kg/m³\n"
                f"α = {alpha_km:.2f} km\n"
                f"wmax = {wmax_km:.3f} km\n"
                f"x0 = {x0_km:.2f} km\n"
                f"xb = {xb_km:.2f} km\n"
                f"x0' = {x0p_km:.2f} km\n"
                f"wb = {wb_km:.3f} km"
            )

            ax.text(
                0.98, 0.97, texto,
                transform=ax.transAxes,
                fontsize=10,
                verticalalignment='top',
                horizontalalignment='right',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.85)
            )
            
        ax.relim()              # Recalcula límites con los nuevos datos
        ax.autoscale_view()     # Ajusta automáticamente X y Y
        
        plt.tight_layout()
        plt.show()

## Reinicio y visualización de cambios

In [7]:
# -----------------------------
# Botón de reinicio
# -----------------------------
def resetear(b):
    slider_D.value = 1e23
    slider_V0.value = 1e13
    slider_rho_m.value = 3300
    slider_rho_l.value = 2600
    slider_xmax.value = 600
    slider_npts.value = 1200
    check_forebulge.value = True
    check_sombreado.value = True
    check_texto.value = True
    actualizar()

boton_reset.on_click(resetear)

In [8]:
# -----------------------------
# Visualización de cambios
# -----------------------------
for w in [slider_D, slider_V0, slider_rho_m, slider_rho_l, slider_xmax, slider_npts,
          check_forebulge, check_sombreado, check_texto]:
    w.observe(actualizar, names='value')

## Despliegue

In [9]:
# -----------------------------
# Diseño del panel
# -----------------------------
titulo = widgets.HTML("<h3>Simulador interactivo de flexura litosférica</h3>")
subtitulo = widgets.HTML("<b>Controla los parámetros físicos y geométricos del modelo</b>")

panel_izquierdo = widgets.VBox([
    titulo,
    subtitulo,
    slider_D,
    slider_V0,
    slider_rho_m,
    slider_rho_l,
    slider_xmax,
    slider_npts,
    widgets.HBox([check_forebulge, check_sombreado]),
    check_texto,
    boton_reset
])

ui = widgets.HBox([
    panel_izquierdo
])

display(ui, out)

actualizar()

Output()